# Using a custom spectrum in SYOTools

In [ ]:
from syotools.models import Camera, Telescope, Source, SourcePhotometricExposure
import numpy as np
import astropy.units as u 

# The Basics of running an imaging calculation

We will use one of SYOTools' built-in templates, the Orion Nebula

In [ ]:
# create a Telescope
tel = Telescope()
tel.set_from_hwome("EAC5")
# Select an Instrument
#print(tel.instruments)
inst = tel.instruments["HRI_S.HRI_S_NIR_Imager"]

# Create a Source
source = Source() 
redshift = 0. # changes to these are not implemented yet 
extinction = 0.
magnitude = 25.0
template = "Orion Nebula"
# Configure the Source
source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")

# Make an Exposure
exp = SourcePhotometricExposure()
# Add the Source to the Exposure
exp.source = source
# Add the Exposure to the Instrument
inst.add_exposure(exp)

#Configure the Exposure
exp.exptime = 60 * u.s
exp.unknown = "snr"

print("Initial Template:", template)
print("SNR:", exp.snr)

But what's available in the default SYOTools spectra library? Let's look.

The library, and the functions that create it, can be loaded separately. The library is a dictionary of Synphot SourceSpectrum objects, whose input files are stored in the SYOTools repo (under data/)

In [ ]:
from syotools.spectra.spec_defaults import syn_spectra_library

print(syn_spectra_library.keys())

# Add a new spectrum to the library.

We will use the same functions SYOTools uses to build the library in the first place.

In [ ]:
import os
from syotools.spectra.utils import load_synfits, load_fesc, load_txtfile

## Example 1: A FITS file

In [ ]:
# FITS File from the STScI TRDS Reference Atlas collection: https://archive.stsci.edu/hlsp/reference-atlases
data_path = os.path.abspath(os.path.join('..','common', 'star_galaxy'))

spectrum = {'desc': '18 Sco',
'file': [data_path, '18sco_stis_006.fits'],
'band': 'johnson,v'}

syn_spectra_library[spectrum["desc"]] = load_txtfile(spectrum)

## Example 2: A text file

In [ ]:
data_path = os.path.abspath(os.path.join('..','common', 'star_galaxy'))

spectrum = {'desc': 'A0V Star',
'file': [data_path, 'pickles_uk_9.ascii'],
'band': 'galex,fuv'}

syn_spectra_library[spectrum["desc"]] = load_txtfile(spectrum)

## Example 3: An analytic spectrum

In [ ]:
import synphot as syn
import stsynphot as stsyn
import numpy as np

# wavelength in Angstroms
wave = np.arange(100, 30000, 300) << u.AA

# Make an 8,000K blackbody
bb = syn.spectrum.SourceSpectrum(syn.models.BlackBody1D, temperature=8000)
# normalize it
bb = bb.normalize(30.0 * u.ABmag, band=stsyn.band('galex,fuv'))
# This library is built on the idea of storing flux and wavelength arrays, so we need to make this an empirical spectrum.
bb = syn.spectrum.SourceSpectrum(syn.models.Empirical1D, points=wave, lookup_table=bb(wave))

# Store the new source template in the array
syn_spectra_library['Blackbody (8,000K)'] = bb

# The Library, Revisited

Our new spectra are in there now.

In [ ]:
print(syn_spectra_library.keys())

# Use the new spectra in calculations

In [ ]:
template = "18 Sco"

source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")
snr = exp.calculate_snr()
print("SNR:", template, exp.snr)

template = "A0V Star"

source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")
snr = exp.calculate_snr()
print("SNR:", template, exp.snr)

template = "Blackbody (8,000K)"

source.set_sed(template, magnitude, redshift, extinction, bandpass="johnson,v")
snr = exp.calculate_snr()
print("SNR:", template, exp.snr)